Task 2:

Loading in and recreate the same train/validation/test split from Week 2 and 3:
- 70% Training
- 15% Validation
- 15% Test
- random_state = 42
- Stratified split to preserve the class distribution

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

#Load the Training Dataset
df = pd.read_csv("community_risk_screening_train.csv")

#Separate Features and Target
feature_columns = [
    "age_at_screening",
    "region",
    "housing_status",
    "prior_crisis_events_12mo",
    "phq2_score",
    "substance_use_flag",
    "primary_support_type",
    "days_since_last_contact"
]

x = df[feature_columns]
y = df["crisis_within_90_days"]

#Split into Training (70%) and Temporary (30%) Data
x_train, x_temp, y_train, y_temp = train_test_split(
    x,y,
    test_size = 0.30,
    random_state = 42,
    stratify = y
)

#Split Temporary Data into Validation (15%) and Test (15%)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp,
    test_size = 0.50,
    random_state = 42,
    stratify = y_temp
)

#Display Dataset Sizes
print("Training Set:", len(x_train))
print("Validation Set:", len(x_val))
print("Testing Set:", len(x_test))

Training Set: 1680
Validation Set: 360
Testing Set: 360


Training the Majority-Class Dummy Baseline

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score, f1_score

#Create the Majority-Class Dummy Model
dummy_model = DummyClassifier(strategy = "most_frequent")

#Train the Model Using the Training Data
dummy_model.fit(x_train, y_train)

#Make Predictions on the Validation Set
dummy_predictions = dummy_model.predict(x_val)

#Calculate Validation Metrics
dummy_recall = recall_score(
    y_val, dummy_predictions,
    pos_label = "crisis",
    zero_division = 0
)

dummy_f1 = f1_score(
    y_val, dummy_predictions,
    pos_label = "crisis",
    zero_division = 0
)

#Display Results
print("Majority-Class Dummy Results")
print("Predicted Class:", pd.Series(dummy_predictions).mode()[0])
print("Recall:", round(dummy_recall, 3))
print("F1 Score:", round(dummy_f1, 3))

Majority-Class Dummy Results
Predicted Class: no_crisis
Recall: 0.0
F1 Score: 0.0


The first experiment is complete:
| Model | Configuration | Validation Recall | Validation F1 |
|---|---|---:|---:|
| Majority-Class Dummy | `most_frequent` | 0.000 | 0.000 |

This model will always predicts no_crisis, so it fails to identify the crisis cases, establishing our baseline performance to compare with the other models.

Random Forest Preprocessing

Preparing the data without fitting the preprocessing on the validation set.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

#Identify Numeric and Categorical Features
numeric_features = [
    "age_at_screening",
    "prior_crisis_events_12mo",
    "phq2_score",
    "days_since_last_contact"
]

categorical_features = [
    "region",
    "housing_status",
    "substance_use_flag",
    "primary_support_type"
]

#Prepare Numeric Features
numeric_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "median"))])

#Prepare Categorical Features
categorical_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown = "ignore"))
])

#Combine Preprocessing Steps
preprocessor = ColumnTransformer(transformers = [
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Random Forest Preprocessing is Ready!")

Random Forest Preprocessing is Ready!


Random Forest Experiments

In [12]:
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

#Define the Random Forest Search Space
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "class_weight": ["balanced"]
}

#Set up the 5 Fold Stratified Cross Validation
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42,
)

#Define Recall and F1 Score
scoring = {
    "recall" : make_scorer(recall_score, pos_label = "crisis", zero_division = 0),
    "f1" : make_scorer(f1_score, pos_label = "crisis", zero_division = 0)
}

#Store Every Experiment's Results
random_forest_results = []

#Run Every Hyperparameter Configuration
for config in ParameterGrid(param_grid):

    #Create the Random Forest Pipeline
    rf_pipeline = Pipeline(steps = [("preprocessor", preprocessor), ("model", RandomForestClassifier(random_state = 42, **config))
])

    #Perform Cross - Validation on Training Data
    cv_results = cross_validate(
        rf_pipeline,
        x_train,
        y_train,
        cv = cv,
        scoring = scoring,
        n_jobs = -1,
    )

    #Train on the Full Training Split
    rf_pipeline.fit(x_train, y_train)

    #Evaluate on the Validation Split
    val_predictions = rf_pipeline.predict(x_val)

    val_recall = recall_score(
        y_val,
        val_predictions,
        pos_label = "crisis",
        zero_division = 0
    )

    val_f1 = f1_score(
        y_val,
        val_predictions,
        pos_label = "crisis",
        zero_division = 0
    )

    #record the Configuration and Results
    random_forest_results.append(
        {
            "Model": "Random Forest",
            "Configuration": str(config),
            "CV Recall": round(cv_results["test_recall"].mean(), 3),
            "CV F1": round(cv_results["test_f1"].mean(), 3),
            "Validation Recall": round(val_recall, 3),
            "Validation F1": round(val_f1, 3)
        }
    )

#Display Experiment Results
rf_results_df = pd.DataFrame(random_forest_results)

print("Random Forest Experiment Results:")
print(rf_results_df.to_string(index = False))

Random Forest Experiment Results:
        Model                                                        Configuration  CV Recall  CV F1  Validation Recall  Validation F1
Random Forest    {'class_weight': 'balanced', 'max_depth': 5, 'n_estimators': 100}      0.732  0.752              0.736          0.722
Random Forest    {'class_weight': 'balanced', 'max_depth': 5, 'n_estimators': 200}      0.728  0.750              0.717          0.717
Random Forest   {'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 100}      0.724  0.774              0.717          0.724
Random Forest   {'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 200}      0.712  0.764              0.717          0.731
Random Forest {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 100}      0.720  0.772              0.736          0.736
Random Forest {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 200}      0.712  0.771              0.717          0.717


Random Forest Experiment Results:

All six Random Forest configurations were successfully tested using the 5 fold cross validation on the training data. Their performance was then evaluated using the validation set.

## Experiment Log

| Model | Configuration | CV Recall | CV F1 | Validation Recall | Validation F1 |
|---|---|---:|---:|---:|---:|
| Majority-Class Dummy | `most_frequent` | N/A | N/A | 0.000 | 0.000 |
| Random Forest | Depth 5, 100 trees | 0.732 | 0.752 | 0.736 | 0.722 |
| Random Forest | Depth 5, 200 trees | 0.728 | 0.750 | 0.717 | 0.717 |
| Random Forest | Depth 10, 100 trees | 0.724 | 0.774 | 0.717 | 0.724 |
| Random Forest | Depth 10, 200 trees | 0.712 | 0.764 | 0.717 | 0.731 |
| Random Forest | Unlimited depth, 100 trees | 0.720 | 0.772 | 0.736 | 0.736 |
| Random Forest | Unlimited depth, 200 trees | 0.712 | 0.771 | 0.717 | 0.717 |

The configuration with 100 estimators and unlimited tree depth produced the highest validation F1 score of 0.736, with a recall of 0.736. Even through this configuration improved the F1 score compare to the one for Logistic Regression baseline, the recall was lower. This tells us that there is still room for improvement  in identifying potential crisis cases.

Gradient Boosting Experiments

In [14]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import ParameterGrid, cross_validate
from sklearn.metrics import make_scorer, recall_score, f1_score
from sklearn.pipeline import Pipeline
import pandas as pd

#Gradient Boosting Configurations
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1],
    "max_depth": [2, 3, 5]
}

#Use the Same Scoring Metrics as the Random Forest Experiments
scoring = {
    "recall": make_scorer(recall_score, pos_label = "crisis", zero_division = 0),
    "f1": make_scorer(f1_score, pos_label = "crisis", zero_division = 0)
}

gradient_boosting_results = []

#Run Every Configuration
for config in ParameterGrid(param_grid):
    gb_pipeline = Pipeline(steps = [
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state = 42, **config))
    ])

    #5 Fold Cross Validation Using Training Data
    cv_results = cross_validate(
        gb_pipeline,
        x_train,
        y_train,
        cv = cv,
        scoring = scoring,
        n_jobs = -1,
    )

    #Train on the Full Training Split and Evaluate on Validation Data
    gb_pipeline.fit(x_train, y_train)
    val_predictions = gb_pipeline.predict(x_val)

    val_recall = recall_score(
        y_val, val_predictions, pos_label = "crisis", zero_division = 0
    )
    val_f1 = f1_score(y_val, val_predictions, pos_label = "crisis", zero_division = 0)

    #Save Experiment Results
    gradient_boosting_results.append(
        {
            "Model": "Gradient Boosting",
            "Configuration": str(config),
            "CV Recall": round(cv_results["test_recall"].mean(), 3),
            "CV F1": round(cv_results["test_f1"].mean(), 3),
            "Validation Recall": round(val_recall, 3),
            "Validation F1": round(val_f1, 3)
        }
    )

#Display All 18 Configurations
gb_results_df = pd.DataFrame(gradient_boosting_results)
print("Gradient Boosting Experiment Results:")
print(gb_results_df.to_string(index = False))

Gradient Boosting Experiment Results:
            Model                                                Configuration  CV Recall  CV F1  Validation Recall  Validation F1
Gradient Boosting  {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 50}      0.000  0.000              0.000          0.000
Gradient Boosting {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100}      0.411  0.562              0.453          0.608
Gradient Boosting {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200}      0.521  0.664              0.566          0.706
Gradient Boosting  {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}      0.000  0.000              0.000          0.000
Gradient Boosting {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}      0.525  0.670              0.528          0.675
Gradient Boosting {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}      0.573  0.705              0.623          0.750
Gradient Boosting  {'learning_rate': 0.01, 'm

Gradient Boosting Experiment Results:

All 18 Gradient Boosting configurations were tested using the 5 fold cross validation on the training data. Each configuration was then evaluated using the validation set with both recall and F1 scores were recorded.

### Experiment Results

| Model | Configuration | CV Recall | CV F1 | Validation Recall | Validation F1 |
|---|---|---:|---:|---:|---:|
| Gradient Boosting | LR 0.01, Depth 2, 50 estimators | 0.000 | 0.000 | 0.000 | 0.000 |
| Gradient Boosting | LR 0.01, Depth 2, 100 estimators | 0.411 | 0.562 | 0.453 | 0.608 |
| Gradient Boosting | LR 0.01, Depth 2, 200 estimators | 0.521 | 0.664 | 0.566 | 0.706 |
| Gradient Boosting | LR 0.01, Depth 3, 50 estimators | 0.000 | 0.000 | 0.000 | 0.000 |
| Gradient Boosting | LR 0.01, Depth 3, 100 estimators | 0.525 | 0.670 | 0.528 | 0.675 |
| Gradient Boosting | LR 0.01, Depth 3, 200 estimators | 0.573 | 0.705 | 0.623 | 0.750 |
| Gradient Boosting | LR 0.01, Depth 5, 50 estimators | 0.000 | 0.000 | 0.000 | 0.000 |
| Gradient Boosting | LR 0.01, Depth 5, 100 estimators | 0.577 | 0.712 | 0.623 | 0.750 |
| Gradient Boosting | LR 0.01, Depth 5, 200 estimators | 0.618 | 0.741 | 0.642 | 0.747 |
| Gradient Boosting | LR 0.1, Depth 2, 50 estimators | 0.634 | 0.749 | 0.660 | 0.769 |
| Gradient Boosting | LR 0.1, Depth 2, 100 estimators | 0.647 | 0.760 | 0.679 | 0.783 |
| Gradient Boosting | LR 0.1, Depth 2, 200 estimators | 0.651 | 0.757 | 0.679 | 0.791 |
| Gradient Boosting | LR 0.1, Depth 3, 50 estimators | 0.647 | 0.757 | 0.660 | 0.761 |
| Gradient Boosting | LR 0.1, Depth 3, 100 estimators | 0.655 | 0.755 | 0.660 | 0.753 |
| Gradient Boosting | LR 0.1, Depth 3, 200 estimators | 0.663 | 0.749 | 0.660 | 0.753 |
| Gradient Boosting | LR 0.1, Depth 5, 50 estimators | 0.655 | 0.758 | 0.660 | 0.745 |
| Gradient Boosting | LR 0.1, Depth 5, 100 estimators | 0.663 | 0.763 | 0.660 | 0.753 |
| Gradient Boosting | LR 0.1, Depth 5, 200 estimators | 0.655 | 0.757 | 0.660 | 0.737 |

*LR = Learning Rate. Depth = Maximum Tree Depth.*

The Gradient Boosting experiments show us that the configurations using a learning rate of 0.1 generally performed better than those using 0.01. The configuration with a learning rate of 0.1, maximum depth of 2, and 200 estimators has achieved the highest validation F1 score of 0.791 and a recall of 0.679. Even through this configuration improved the F1 score compared to both the Random Forest and Logistic Regression results. The recall was still lower so more evalution is needed before selecting the final model.

### Comparison Against Success Criteria

The success criteria established in Task 1 were:

- Recall: At least 0.80
- F1 Score: At least 0.62

Each model configuration was compared against these thresholds using its validation results.

| Model | Configuration | Validation Recall | Validation F1 | Meets Both Criteria? |
|---|---|---:|---:|---|
| Logistic Regression | Week 2 Baseline | 0.774 | 0.617 | No |
| Majority-Class Dummy | Most Frequent | 0.000 | 0.000 | No |
| Random Forest | Depth 5, 100 estimators | 0.736 | 0.722 | No |
| Random Forest | Depth 5, 200 estimators | 0.717 | 0.717 | No |
| Random Forest | Depth 10, 100 estimators | 0.717 | 0.724 | No |
| Random Forest | Depth 10, 200 estimators | 0.717 | 0.731 | No |
| Random Forest | Unlimited Depth, 100 estimators | 0.736 | 0.736 | No |
| Random Forest | Unlimited Depth, 200 estimators | 0.717 | 0.717 | No |
| Gradient Boosting | LR 0.01, Depth 2, 50 estimators | 0.000 | 0.000 | No |
| Gradient Boosting | LR 0.01, Depth 2, 100 estimators | 0.453 | 0.608 | No |
| Gradient Boosting | LR 0.01, Depth 2, 200 estimators | 0.566 | 0.706 | No |
| Gradient Boosting | LR 0.01, Depth 3, 50 estimators | 0.000 | 0.000 | No |
| Gradient Boosting | LR 0.01, Depth 3, 100 estimators | 0.528 | 0.675 | No |
| Gradient Boosting | LR 0.01, Depth 3, 200 estimators | 0.623 | 0.750 | No |
| Gradient Boosting | LR 0.01, Depth 5, 50 estimators | 0.000 | 0.000 | No |
| Gradient Boosting | LR 0.01, Depth 5, 100 estimators | 0.623 | 0.750 | No |
| Gradient Boosting | LR 0.01, Depth 5, 200 estimators | 0.642 | 0.747 | No |
| Gradient Boosting | LR 0.1, Depth 2, 50 estimators | 0.660 | 0.769 | No |
| Gradient Boosting | LR 0.1, Depth 2, 100 estimators | 0.679 | 0.783 | No |
| Gradient Boosting | LR 0.1, Depth 2, 200 estimators | 0.679 | 0.791 | No |
| Gradient Boosting | LR 0.1, Depth 3, 50 estimators | 0.660 | 0.761 | No |
| Gradient Boosting | LR 0.1, Depth 3, 100 estimators | 0.660 | 0.753 | No |
| Gradient Boosting | LR 0.1, Depth 3, 200 estimators | 0.660 | 0.753 | No |
| Gradient Boosting | LR 0.1, Depth 5, 50 estimators | 0.660 | 0.745 | No |
| Gradient Boosting | LR 0.1, Depth 5, 100 estimators | 0.660 | 0.753 | No |
| Gradient Boosting | LR 0.1, Depth 5, 200 estimators | 0.660 | 0.737 | No |

Summary:

None of the tested configurations met any of the success criteria. Several Random Forest and Gradient Boosting configurations achieved the target F1 score, but none of them reached the required recall of 0.80. This indicates that identifying more potential crisis cases remains the main area for improvement.

Classification Threshold Tuning

In [16]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, f1_score, precision_score
import pandas as pd

#Best Random Forest Configuration
best_rf_pipeline = Pipeline(steps = [("preprocessor", preprocessor), ("model", RandomForestClassifier(
    n_estimators = 100,
    max_depth = None,
    class_weight = "balanced",
    random_state = 42,
    ))
])

#Best Gradient Boosting Configuration
best_gb_pipeline = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators = 200,
        max_depth = 2,
        learning_rate = 0.1,
        random_state = 42,
    ))
])

#Train Both Models Using Training Data Only
best_rf_pipeline.fit(x_train, y_train)
best_gb_pipeline.fit(x_train, y_train)

#Get Crisis Probabilities from the Validation Set
rf_probabilities = best_rf_pipeline.predict_proba(x_val)
gb_probabilities = best_gb_pipeline.predict_proba(x_val)

#Find the Column Representing the Crisis Class
rf_crisis_index = list(best_rf_pipeline.classes_).index("crisis")
gb_crisis_index = list(best_gb_pipeline.classes_).index("crisis")

rf_crisis_probabilities = rf_probabilities[:, rf_crisis_index]
gb_crisis_probabilities = gb_probabilities[:, gb_crisis_index]

#Test Different Classification Thresholds
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []

for model_name, probabilities in [
    ("Random Forest", rf_crisis_probabilities),
    ("Gradient Boosting", gb_crisis_probabilities)
]:

    for threshold in thresholds:

        predictions = [
            "crisis" if probability >= threshold else "no_crisis"
            for probability in probabilities
        ]

        recall = recall_score(
            y_val, predictions, pos_label="crisis", zero_division=0
        )

        f1 = f1_score(
            y_val, predictions, pos_label="crisis", zero_division=0
        )

        precision = precision_score(
            y_val, predictions, pos_label="crisis", zero_division=0
        )

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Recall": round(recall, 3),
            "Precision": round(precision, 3),
            "F1": round(f1, 3),
        })

#Display all Threshold Experiments
threshold_results_df = pd.DataFrame(threshold_results)

print("Classification Threshold Tuning Results:")
print(threshold_results_df.to_string(index = False))

Classification Threshold Tuning Results:
            Model  Threshold  Recall  Precision    F1
    Random Forest       0.20   0.792      0.350 0.486
    Random Forest       0.25   0.792      0.467 0.587
    Random Forest       0.30   0.792      0.532 0.636
    Random Forest       0.35   0.736      0.600 0.661
    Random Forest       0.40   0.736      0.650 0.690
    Random Forest       0.45   0.736      0.709 0.722
    Random Forest       0.50   0.736      0.736 0.736
Gradient Boosting       0.20   0.736      0.696 0.716
Gradient Boosting       0.25   0.717      0.717 0.717
Gradient Boosting       0.30   0.698      0.755 0.725
Gradient Boosting       0.35   0.698      0.881 0.779
Gradient Boosting       0.40   0.698      0.902 0.787
Gradient Boosting       0.45   0.679      0.900 0.774
Gradient Boosting       0.50   0.679      0.947 0.791


Classification Threshold Tuning Results

Using the best Random Forest and Gradient Boosting configurations were tested using seven different classification thresholds. The purpose of this is to determine whether adjusting the threshold could improve recall while maintaining an F1 score of at least 0.62.

## Experiment Log

| Model | Threshold | Recall | Precision | F1 |
|---|---:|---:|---:|---:|
| Random Forest | 0.20 | 0.792 | 0.350 | 0.486 |
| Random Forest | 0.25 | 0.792 | 0.467 | 0.587 |
| Random Forest | 0.30 | 0.792 | 0.532 | 0.636 |
| Random Forest | 0.35 | 0.736 | 0.600 | 0.661 |
| Random Forest | 0.40 | 0.736 | 0.650 | 0.690 |
| Random Forest | 0.45 | 0.736 | 0.709 | 0.722 |
| Random Forest | 0.50 | 0.736 | 0.736 | 0.736 |
| Gradient Boosting | 0.20 | 0.736 | 0.696 | 0.716 |
| Gradient Boosting | 0.25 | 0.717 | 0.717 | 0.717 |
| Gradient Boosting | 0.30 | 0.698 | 0.755 | 0.725 |
| Gradient Boosting | 0.35 | 0.698 | 0.881 | 0.779 |
| Gradient Boosting | 0.40 | 0.698 | 0.902 | 0.787 |
| Gradient Boosting | 0.45 | 0.679 | 0.900 | 0.774 |
| Gradient Boosting | 0.50 | 0.679 | 0.947 | 0.791 |

Summary:

The threshold tuning experiments showed that lowering the classification threshold for Random Forest increased recall. With a threshold of 0.30, it achieved a recall of 0.792 and an F1 score of 0.636. The F1 score target has been met however the recall was slightly below the required recall of 0.80.

While for the Gradient Boosting it had maintained a higher F1 scores across the tested thresholds, but its recall remained below the target.

None of the tested thresholds achieved the success criteria of at least 0.80 recall and 0.62 F1 score.

XGBoost Model Training

In [3]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid, cross_validate
from sklearn.metrics import make_scorer, recall_score, f1_score

#Define the XGBoost Configurations to Test Out
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.1]
}

print("Number of Configurations:", len(list(ParameterGrid(param_grid))))

Number of Configurations: 8


In [9]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score, f1_score
import pandas as pd

#Use 5 Fold Cross Validation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Define Evaluation Metrics
scoring ={
    "recall": make_scorer(
        recall_score, pos_label="crisis", zero_division=0
    ),
    "f1": make_scorer(
        f1_score, pos_label="crisis", zero_division=0
    )
}

#Store Results
xgb_results = []

In [17]:
#Convert Target Labels to Numeric Values
y_train_xgb = (y_train == "crisis").astype(int)
y_val_xgb = (y_val == "crisis").astype(int)

#Update the Score to Identify Crisis Cases
scoring = {
    "recall": make_scorer(
        recall_score, pos_label=1, zero_division=0
    ),
    "f1": make_scorer(
        f1_score, pos_label=1, zero_division=0
    )
}

#Test all 8 XGBoost Configurations
for config in ParameterGrid(param_grid):

    xgb_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1,
            **config
        ))
    ])

    #Perform 5 Fold Cross Validation on Training Data
    cv_results = cross_validate(
        xgb_pipeline,
        x_train,
        y_train_xgb,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    #Train the Model on the Full Training Split
    xgb_pipeline.fit(x_train, y_train_xgb)

    #Evaluate the Validation Split
    val_predictions = xgb_pipeline.predict(x_val)
    val_recall = recall_score(
    y_val_xgb, val_predictions, zero_division=0
    )
    val_f1 = f1_score(y_val_xgb, val_predictions, zero_division=0)

    #Save Results
    xgb_results.append({
        "Model": "XGBoost",
        "Configuration": str(config),
        "CV Recall": round(cv_results["test_recall"].mean(), 3),
        "CV F1": round(cv_results["test_f1"].mean(), 3),
        "Validation Recall": round(val_recall, 3),
        "Validation F1": round(val_f1, 3)
    })

#Display Results
xgb_results_df = pd.DataFrame(xgb_results)
print("XGBoost Experiment Results:")
print(xgb_results_df.to_string(index=False))

XGBoost Experiment Results:
  Model                                                Configuration  CV Recall  CV F1  Validation Recall  Validation F1
XGBoost {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}      0.488  0.641              0.491          0.650
XGBoost {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}      0.590  0.723              0.604          0.744
XGBoost {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100}      0.537  0.678              0.585          0.721
XGBoost {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200}      0.630  0.748              0.660          0.769
XGBoost  {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}      0.655  0.766              0.679          0.791
XGBoost  {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}      0.671  0.776              0.660          0.753
XGBoost  {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}      0.679  0.781              0.679          0.758
XGBo

XGBoost Experiment Results:

All 8 of the XGBoost configurations were successfully tested using the 5 fold cross validation on the training data.

The best results was the F1 score with a 0.791 that was achieved with a learning rate of 0.1, maximum depth of 3 and 100 estimators. However, it has achieved a recall of 0.679.

Even though through this test the configurations have achieved an F1 score above the targeted goal of 0.62, but non reached the wanted recall target of 0.80. The highest recall we got is 0.679.

The results from the XGBoost provides us a strong F1 score but further threshold tuning is needed to see if the recall can also improve while maintaining the acceptable F1 score.

XGBoost Threshold Tuning

In [22]:
#Training the Best XGBoost Configuration
best_xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        learning_rate=0.1,
        max_depth=3,
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

#Train Using the Training Data
best_xgb_pipeline.fit(x_train, y_train_xgb)

#Get Crisis Prediction Probabilities
xgb_probabilities = best_xgb_pipeline.predict_proba(x_val)[:, 1]

#Test Different Thresholds
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

xgb_threshold_results = []

for threshold in thresholds:

    #Convert Probabilities into Predictions
    predictions = (xgb_probabilities >= threshold).astype(int)

    #Calculate Recall and F1
    recall = recall_score(
        y_val_xgb, predictions, zero_division=0
    )
    f1 = f1_score(
        y_val_xgb, predictions, zero_division=0
    )

    #Save Results
    xgb_threshold_results.append({
        "Model": "XGBoost",
        "Threshold": threshold,
        "Recall": round(recall, 3),
        "F1": round(f1, 3)
    })

#Display Results
xgb_threshold_df = pd.DataFrame(xgb_threshold_results)
print("XGBoost Threshold Tuning Results:")
print(xgb_threshold_df.to_string(index=False))

XGBoost Threshold Tuning Results:
  Model  Threshold  Recall    F1
XGBoost       0.20   0.736 0.716
XGBoost       0.25   0.736 0.750
XGBoost       0.30   0.698 0.725
XGBoost       0.35   0.698 0.740
XGBoost       0.40   0.698 0.740
XGBoost       0.45   0.679 0.774
XGBoost       0.50   0.679 0.791


XGBoost Threshold Tuning Results:

7 classification thresholds were tested on the validation set using the best XGBoost configuration we got before.

The highest recall we got is 0.736 at a thresholds of 0.20 and 0.25. The highest F1 score we got is 0.791at the threshold of 0.50.

Lowering the threshold improved the recall but none have reached the targeted recall of 0.80. While all 7 thresholds achieved the F1 score above the targeted F1 of 0.62.

If we compare with the selected Random Forest model, the XGBoost achieved a higher F1 score but lower recall. So we will still be using the Random Forest at a threshold of 0.30 as the preferred model because of having the stronger recall while also maintaining an high F1 score above the target.